# ============================================
# 02_uci_electricity_load_diagrams_downloader.ipynb
# UCI ElectricityLoadDiagrams20112014
# ============================================

## Dataset Title
UCI ElectricityLoadDiagrams20112014

## Source URL
Official dataset page:  
https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014

Direct download URL:  
https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip

## Purpose of This Dataset
This dataset is used for baseline forecasting and demand-pattern modeling.

It is suitable for:
- time-series preprocessing
- load forecasting baselines
- multivariate demand analysis
- benchmarking sequence models
- feature engineering for downstream optimization

## Why We Need This Dataset
We need this dataset because it provides a large, structured, real electricity demand dataset with:
- 370 clients
- 15-minute resolution
- no missing values
- long historical sequence length

This makes it highly useful for:
- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset-specific folders
3. Download the real dataset ZIP from the official source
4. Extract the downloaded ZIP
5. Identify usable files
6. Save a dataset manifest
7. Build a clean ZIP package for reuse
8. Save everything in Google Drive
9. Download the complete package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust

## Expected Outputs
At the end of this notebook, we will have:

- raw dataset ZIP
- extracted data files
- dataset manifest JSON
- clean bundle ZIP
- full package ZIP

## Notes
According to the official UCI page, the main data file is `LD2011_2014.txt`, values are recorded in kW, each column corresponds to one client, and timestamps are stored in `yyyy-mm-dd hh:mm:ss` format. :contentReference[oaicite:1]{index=1}

In [1]:
# ============================================
# 02_uci_electricity_load_diagrams_downloader.ipynb
# COMPLETE PYTHON CODE
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import requests

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "UCI ElectricityLoadDiagrams20112014"
SOURCE_PAGE = "https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014"
DOWNLOAD_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
from google.colab import drive, files
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "02_uci_electricity_load_diagrams"
raw_dir = dataset_root / "raw"
extracted_dir = dataset_root / "extracted"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, extracted_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", extracted_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download dataset ZIP
# -----------------------------
filename = os.path.basename(urlparse(DOWNLOAD_URL).path)
download_path = raw_dir / filename

if not download_path.exists():
    print("\nDownloading dataset ZIP...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

# -----------------------------
# 6. Verify file size
# -----------------------------
if download_path.exists():
    size_mb = download_path.stat().st_size / (1024 * 1024)
    print(f"Downloaded ZIP size: {size_mb:.2f} MB")
else:
    raise FileNotFoundError("Dataset ZIP was not downloaded.")

# -----------------------------
# 7. Extract ZIP
# -----------------------------
print("\nExtracting ZIP...")
with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extracted_dir)

print("Extraction complete:", extracted_dir)

# -----------------------------
# 8. List extracted files
# -----------------------------
all_files = sorted([p for p in extracted_dir.rglob("*") if p.is_file()])
usable_files = [
    p for p in all_files
    if p.suffix.lower() in [".txt", ".csv", ".data", ".json", ".xlsx", ".xls"]
]

print("\nTotal extracted files:", len(all_files))
print("Usable files:", len(usable_files))
for fp in usable_files:
    print(" -", fp)

# -----------------------------
# 9. Save manifest JSON
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(download_path),
    "extracted_dir": str(extracted_dir),
    "all_files": [str(p) for p in all_files],
    "usable_files": [str(p) for p in usable_files],
    "used_in_next_notebooks": [
        "Notebook 2: Baseline Models",
        "Notebook 3: NEO-Forecast",
        "Notebook 6: NEO-Opt",
        "Notebook 7: NEO-Robust"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 10. Create clean bundle ZIP
# -----------------------------
bundle_zip_path = bundle_dir / "uci_electricity_load_diagrams_usable_files.zip"

with zipfile.ZipFile(bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in usable_files:
        arcname = fp.relative_to(extracted_dir)
        zf.write(fp, arcname)

print("Created bundle ZIP:", bundle_zip_path)

# -----------------------------
# 11. Create full package ZIP
# -----------------------------
full_zip_path = dataset_root / "uci_electricity_load_diagrams_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:

    # raw zip
    zf.write(download_path, arcname="raw/" + download_path.name)

    # extracted files
    for file in extracted_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname="extracted/" + str(file.relative_to(extracted_dir)))

    # bundle zip
    zf.write(bundle_zip_path, arcname="bundle/" + bundle_zip_path.name)

    # manifest
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP:", full_zip_path)

# -----------------------------
# 12. Optional runtime copy
# -----------------------------
runtime_copy = Path("/content/uci_electricity_load_diagrams_manifest_copy.json")
shutil.copy2(manifest_path, runtime_copy)
print("Runtime copy created:", runtime_copy)

# -----------------------------
# 13. Summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root      :", dataset_root)
print("Raw ZIP           :", download_path)
print("Extracted folder  :", extracted_dir)
print("Manifest          :", manifest_path)
print("Bundle ZIP        :", bundle_zip_path)
print("Full Package ZIP  :", full_zip_path)
print("===================")

# -----------------------------
# 14. Download full package to PC
# -----------------------------
print("\nStarting PC download of full package ZIP...")
files.download(str(full_zip_path))

Dataset Name : UCI ElectricityLoadDiagrams20112014
Source Page  : https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014
Download URL : https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams
 - /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/extracted
 - /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/bundle

Using cached file: /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/raw/electricityloaddiagrams20112014.zip
Downloaded ZIP size: 249.23 MB

Extracting ZIP...
Extraction complete: /content/drive/MyDrive/PhD_T

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ============================================
# FINAL FORCE SAVE + DOWNLOAD (WORKS 100%)
# ============================================

from google.colab import files
from pathlib import Path
import zipfile
import shutil

print("===== FIXING & DOWNLOADING PACKAGE =====\n")

# -----------------------------
# 1. Ensure folders exist
# -----------------------------
dataset_root.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2. FORCE RECREATE FULL ZIP
# -----------------------------
full_zip_path = dataset_root / "uci_electricity_load_diagrams_FULL_PACKAGE.zip"

print("Creating FULL package ZIP...")

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:

    # Add raw ZIP
    if download_path.exists():
        zf.write(download_path, arcname="raw/" + download_path.name)

    # Add extracted files
    for file in extracted_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname="extracted/" + str(file.relative_to(extracted_dir)))

    # Add bundle ZIP
    if bundle_zip_path.exists():
        zf.write(bundle_zip_path, arcname="bundle/" + bundle_zip_path.name)

    # Add manifest
    if manifest_path.exists():
        zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("ZIP created:", full_zip_path)

# -----------------------------
# 3. VERIFY GOOGLE DRIVE SAVE
# -----------------------------
if full_zip_path.exists():
    print("\nSaved in Google Drive ✅")
    print("Location:", full_zip_path)
    print("Size:", round(full_zip_path.stat().st_size / (1024*1024), 2), "MB")
else:
    raise Exception("ZIP NOT SAVED IN DRIVE ❌")

# -----------------------------
# 4. FORCE COPY (extra safety)
# -----------------------------
backup_path = Path("/content/uci_backup_package.zip")
shutil.copy2(full_zip_path, backup_path)

print("\nBackup copy created:", backup_path)

# -----------------------------
# 5. FORCE DOWNLOAD TO PC
# -----------------------------
print("\nDownloading to your PC...")

files.download(str(backup_path))

print("\n===== DONE SUCCESSFULLY =====")

===== FIXING & DOWNLOADING PACKAGE =====

Creating FULL package ZIP...
ZIP created: /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/uci_electricity_load_diagrams_FULL_PACKAGE.zip

Saved in Google Drive ✅
Location: /content/drive/MyDrive/PhD_Thesis_Datasets/02_uci_electricity_load_diagrams/uci_electricity_load_diagrams_FULL_PACKAGE.zip
Size: 747.84 MB

Backup copy created: /content/uci_backup_package.zip



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


===== DONE SUCCESSFULLY =====
